# Import Libraries

Enable automatic reloading of modules during interactive development:

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

import statsmodels.api as sm # type: ignore
from statsmodels.api import Logit # type: ignore

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling3_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
)

In [2]:

target = 'splashing'
target_set = {'splashing', 'no_fragmentation'}

train, test = get_train_test(
    dataset_filename='df_dimless',
    target=target,
    path_data=Path('..', 'data'),
    target_set=target_set
)

features_to_drop = (
    'Re', 
    'We', 
    'init_volume_fraction',
    'particle_droplet_diameter_ratio', 
    'sedimentation_Re',
    # 'particle_liquid_density_ratio',
    'sedimentation_Stk'
    # 'sign_sedimentation_Re',
    # 'volume_fraction', 
    # 'relative_roughness', 
    # 'inclination',
    # 'wettability',
)

minmax_features = (
    'inclination',
    'volume_fraction',
)

passthrough_features = (
    'wettability',
)

# train = _prepare_df(train, features_to_drop=features_to_drop)
# test = _prepare_df(test, features_to_drop=features_to_drop)

features = list(train.columns)
features.remove(target)

X = train.drop(target, axis=1)
y = train[target].reset_index(drop=True)

estimator = StatsModelsEstimator(Logit)

pipe = _create_pipeline(
    source_features=features,
    minmax_features=minmax_features,
    passthrough_features=passthrough_features,
    features_to_drop=features_to_drop,
    estimator=estimator,
    add_df_transformer=True,
    log_sedimentation_Stk=False,
    add_const=False,
    verbose=False
)
display(pipe)

# pipe.fit(X, y, model__fit_method='fit_regularized')
pipe.fit(X, y)

print(estimator.results_.summary())

No df was gotten.
Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx


Pipeline(steps=[('init_transformer',
                 InitialTransformer(log_roughness=True,
                                    log_sedimentation_Stk=False)),
                ('column_transformer',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  ('inclination',
                                                   'volume_fraction')),
                                                 ('std', StandardScaler(),
                                                  ('sign_sedimentation_Re',
                                                   'particle_liquid_density_ratio',
                                                   'relative_roughness', 'K')),
                                                 ('passthrough', 'passthrough',
                                                  ('wettability',))])),
                ('df_transformer',
                 DataFrameTransformer(add_const=False,
                                      feature_names=['inclination',
                                                     'volume_fraction',
                                                     'sign_sedimentation_Re',
                                                     'particle_liquid_density_ratio',
                                                     'relative_roughness', 'K',
                                                     'wettability'])),
                ('estimator',
                 StatsModelsEstimator(model_class=<class 'statsmodels.discrete.discrete_model.Logit'>))])

Optimization terminated successfully.
         Current function value: 0.384420
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:              splashing   No. Observations:                  297
Model:                          Logit   Df Residuals:                      290
Method:                           MLE   Df Model:                            6
Date:                Sun, 06 Oct 2024   Pseudo R-squ.:                  0.4082
Time:                        19:04:24   Log-Likelihood:                -114.17
converged:                       True   LL-Null:                       -192.93
Covariance Type:            nonrobust   LLR p-value:                 1.983e-31
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
inclination                       2.9317      0.637      4.602      0.

Best result:

In [3]:
y_pred_proba = pipe.predict_proba(
    X=test.drop(target, axis=1)
)

def get_prediction(y_pred_proba):
    y_pred = np.zeros_like(y_pred_proba)
    y_pred[y_pred_proba>0.5] = 1
    
    return y_pred

y_pred = get_prediction(y_pred_proba)

f1_score(test[target], y_pred)

0.8712871287128713

In [4]:
accuracy_score(test[target], y_pred)

0.8266666666666667

In [5]:
roc_auc_score(
    test[target],
    y_pred_proba
)

0.8796296296296297

Idea: Consider log(relative_roughness) - DONE!

With/without particle_liquid_density_ratio

with - strange influence, but higher metrics